In [6]:
plantdoc_train_path = "/media/data/minhht/moe_plantdeasse/data/plantdoc_dataset/train"
plantdoc_test_path = "/media/data/minhht/moe_plantdeasse/data/plantdoc_dataset/test"

In [7]:
import os

print("Train classes:", os.listdir(plantdoc_train_path))
print("Test classes:", os.listdir(plantdoc_test_path))

Train classes: ['Tomato mold leaf', 'Peach leaf', 'Tomato leaf yellow virus', 'Tomato leaf mosaic virus', 'grape leaf black rot', 'Blueberry leaf', 'Strawberry leaf', 'Cherry leaf', 'Tomato leaf bacterial spot', 'Tomato Early blight leaf', 'Apple Scab Leaf', 'Soyabean leaf', 'Tomato leaf', 'Corn rust leaf', 'Corn leaf blight', 'Raspberry leaf', 'Apple leaf', 'Tomato Septoria leaf spot', 'Bell_pepper leaf spot', 'Apple rust leaf', 'grape leaf', 'Potato leaf late blight', 'Potato leaf early blight', 'Squash Powdery mildew leaf', 'Tomato leaf late blight', 'Bell_pepper leaf', 'Corn Gray leaf spot']
Test classes: ['Tomato mold leaf', 'Peach leaf', 'Tomato leaf yellow virus', 'Tomato leaf mosaic virus', 'grape leaf black rot', 'Blueberry leaf', 'Strawberry leaf', 'Cherry leaf', 'Tomato leaf bacterial spot', 'Tomato Early blight leaf', 'Apple Scab Leaf', 'Soyabean leaf', 'Tomato leaf', 'Corn rust leaf', 'Corn leaf blight', 'Raspberry leaf', 'Apple leaf', 'Tomato Septoria leaf spot', 'Bell_pe

In [8]:
from typing import List, Literal
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
from sklearn.model_selection import train_test_split


class LoadDataset(Dataset):
    def __init__(self, data_paths: List[str], split: Literal["train", "validation", "test"], train_ratio: float, transforms=None):
        super().__init__()
        self.data_paths = data_paths
        self.split = split
        self.train_ratio = train_ratio
        self.transforms = transforms
        self.image_paths, self.labels, self.class_to_idx, self.idx_to_class = self._split_data()

    def _load_data(self):
        class_names = sorted(
            [d for d in os.listdir(self.data_paths[0]) if os.path.isdir(os.path.join(self.data_paths[0], d))]
        )
        class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
        idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}
        image_paths = []
        labels = []
        for data_path in self.data_paths:
            for class_name in class_names:
                class_dir = os.path.join(data_path, class_name)
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        image_paths.append(os.path.join(class_dir, img_name))
                        labels.append(class_to_idx[class_name])
        return image_paths, labels, class_to_idx, idx_to_class
    
    def _split_data(self):
        image_paths, labels, class_to_idx, idx_to_class = self._load_data()

        train_paths, temp_paths, train_labels, temp_labels = train_test_split(
            image_paths, labels, test_size= 1-self.train_ratio, stratify=labels, random_state=42, shuffle=True
        )

        val_paths, test_paths, val_labels, test_labels = train_test_split(
            temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42, shuffle=True
        )

        if self.split == 'train':
            return train_paths, train_labels, class_to_idx, idx_to_class
        elif self.split == 'validation':
            return val_paths, val_labels, class_to_idx, idx_to_class
        elif self.split == 'test':
            return test_paths, test_labels, class_to_idx, idx_to_class
        else:
            raise ValueError("split must be 'train', 'validation', or 'test'")

   
    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transforms:
            image = self.transforms(image)
        return image, label

In [9]:
from torchvision import transforms


data_paths = [plantdoc_train_path, plantdoc_test_path]

transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # chuyển PIL → tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = LoadDataset(data_paths, split='train', train_ratio=0.8, transforms=transforms)
val_dataset = LoadDataset(data_paths, split='validation', train_ratio=0.8, transforms=transforms)
test_dataset = LoadDataset(data_paths, split='test', train_ratio=0.8, transforms=transforms)

print("Number of training samples:", len(train_dataset))
print("Number of validation samples:", len(val_dataset))
print("Number of test samples:", len(test_dataset))

Number of training samples: 2041
Number of validation samples: 255
Number of test samples: 256


In [10]:
from torch.utils.data import DataLoader
from torchvision import transforms


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [15]:
import torch
import torch.nn as nn
from torchvision.models import vit_b_16

num_classes = len(train_dataset.class_to_idx)

model = vit_b_16(weights="IMAGENET1K_V1")

# thay head classifier
model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)

In [12]:
# criterion = nn.CrossEntropyLoss()

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=0.001,
# )

# num_epochs = 50
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)

# for epoch in range(num_epochs):
#     model.train()
#     total_loss = 0
#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)
#         optimizer.zero_grad()
#         outputs = model(images)
#         loss = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()
#         total_loss += loss.item()
#     avg_loss = total_loss / len(train_loader)
#     print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

In [13]:
# torch.save(model.state_dict(), "vit_b16_plant.pth")

In [14]:
checkpoint_path = "/media/data/minhht/moe_plantdeasse/notebook/vit_b16_plant.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location="cuda"))
# bỏ classifier
model.heads.head = nn.Identity()
model.eval()

RuntimeError: Error(s) in loading state_dict for MobileNetV3:
	Missing key(s) in state_dict: "features.0.0.weight", "features.0.1.weight", "features.0.1.bias", "features.0.1.running_mean", "features.0.1.running_var", "features.1.block.0.0.weight", "features.1.block.0.1.weight", "features.1.block.0.1.bias", "features.1.block.0.1.running_mean", "features.1.block.0.1.running_var", "features.1.block.1.0.weight", "features.1.block.1.1.weight", "features.1.block.1.1.bias", "features.1.block.1.1.running_mean", "features.1.block.1.1.running_var", "features.2.block.0.0.weight", "features.2.block.0.1.weight", "features.2.block.0.1.bias", "features.2.block.0.1.running_mean", "features.2.block.0.1.running_var", "features.2.block.1.0.weight", "features.2.block.1.1.weight", "features.2.block.1.1.bias", "features.2.block.1.1.running_mean", "features.2.block.1.1.running_var", "features.2.block.2.0.weight", "features.2.block.2.1.weight", "features.2.block.2.1.bias", "features.2.block.2.1.running_mean",

In [ ]:
input_dummy = torch.randn(10, 3, 224, 224)
output_dummy = model(input_dummy)
print("Output shape:", output_dummy.shape)

Output shape: torch.Size([10, 768])


In [ ]:
all_features = []
all_labels = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        features = model(images)
        all_features.append(features.cpu())
        all_labels.append(labels.cpu())

In [ ]:
features = torch.cat(all_features)
labels = torch.cat(all_labels)

In [ ]:
features.shape, labels.shape

(torch.Size([2041, 768]), torch.Size([2041]))

In [ ]:
features

tensor([[-0.5596,  0.3513, -0.4061,  ..., -0.5193, -0.0674,  0.3195],
        [ 0.3598, -0.3513,  0.3532,  ...,  0.3256, -0.0114, -0.5849],
        [ 0.7628,  0.3825, -0.2006,  ..., -0.3300, -0.9108, -0.3814],
        ...,
        [ 0.3778, -0.1938,  0.1855,  ...,  0.3996,  0.0979,  0.4198],
        [ 0.8078, -0.0437, -0.7972,  ..., -0.0688,  0.7182, -0.4215],
        [-0.4406, -0.1369,  0.4793,  ..., -0.6859,  0.1820,  0.1046]])

In [ ]:
features.T.shape

torch.Size([768, 2041])

In [ ]:
import skfuzzy as fuzz
import pandas as pd

results = []

for m in [2.0, 3.0]:
    for c in range(2, 11):

        cntr, u, _, _, _, _, fpc = fuzz.cluster.cmeans(
            features.T,
            c=c,
            m=m,
            error=1e-5,
            maxiter=5000,
            seed=42
        )

        # label cluster
        cluster_labels = u.argmax(axis=0)

        results.append({
            "m": m,
            "clusters": c,
            "fpc": fpc,
            "cluster_labels": cluster_labels
        })